# RGB Cal Target SR Analysis

Compare Native-2x, SAA, and SAA+IBP super-resolution results on the ISO 12233 calibration target (red Bayer channel).

SR is performed on the **red Bayer channel** (RGGB → even rows/cols).  
LR: 768 × 1024 (red channel) → HR: 1536 × 2048 (2× upsample).

**ROI 1 — Horizontal bars:** vertical cross-section through horizontal bar groups.

**ROI 2 — Slanted edge MTF:** ESF/LSF/MTF from a diagonal edge.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import zoom as ndi_zoom
from scipy.optimize import curve_fit

%matplotlib inline
plt.rcParams['figure.dpi'] = 130

# --- Pick the session folder ---
SR_DIR = os.path.join('results', 'cal_target_color_tilt0.28000deg_settle20ms')

# Load images (red-channel SR outputs)
lr_mean   = np.array(Image.open(os.path.join(SR_DIR, 'LR_red_mean.png')), dtype=np.float64)
native_2x = np.array(Image.open(os.path.join(SR_DIR, 'native_2x.png')), dtype=np.float64)
saa       = np.array(Image.open(os.path.join(SR_DIR, 'SAA.png')), dtype=np.float64)
saa_ibp   = np.array(Image.open(os.path.join(SR_DIR, 'SAA_IBP.png')), dtype=np.float64)

hr_images = {
    'Native-2x': native_2x,
    'SAA':       saa,
    'SAA+IBP':   saa_ibp,
}

print(f'LR shape : {lr_mean.shape}')
print(f'HR shape : {native_2x.shape}')

In [ ]:
# Full HR image with axes for coordinate reference
fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(native_2x, cmap='gray', interpolation='nearest')
ax.set_xlabel('Column (HR px)')
ax.set_ylabel('Row (HR px)')
ax.set_title('Full Native-2x image (red channel)')
plt.tight_layout()
plt.show()

## ROI 1 — Horizontal bar resolution

Vertical cross-section through horizontal bar groups. The RGB HR image is half the size of mono (1536×2048 vs 3072×4096), so ROI coordinates are scaled accordingly.

Adjust `PROFILE_COL_HR` and `ROI1_ROWS_HR` below based on the image above.

In [ ]:
# ROI 1: vertical cross-section through horizontal bars
# HR coords (1536x2048) — adjust these based on the image above
PROFILE_COL_HR = 1350    # half of mono's 2700
ROI1_ROWS_HR   = slice(620, 780)   # half of mono's 1240-1560

lr_2x  = ndi_zoom(lr_mean, 2, order=3)
titles = ['LR bicubic 2x', 'Native-2x', 'SAA', 'SAA+IBP']
imgs   = [lr_2x, native_2x, saa, saa_ibp]
colors = ['C0', 'C1', 'C2', 'C3']

rows_hr = np.arange(ROI1_ROWS_HR.start, ROI1_ROWS_HR.stop)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: all profiles overlaid
ax = axes[0]
for title, img, c in zip(titles, imgs, colors):
    profile = img[ROI1_ROWS_HR, PROFILE_COL_HR]
    ax.plot(rows_hr, profile, lw=0.8, alpha=0.8, label=title, color=c)
ax.set_xlabel('Row (HR px)')
ax.set_ylabel('Intensity')
ax.set_title(f'Vertical cross-section at col {PROFILE_COL_HR}')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right: zoomed image strip showing the column location
ax = axes[1]
strip_w = 100
col_lo = PROFILE_COL_HR - strip_w // 2
col_hi = PROFILE_COL_HR + strip_w // 2
ax.imshow(lr_2x[ROI1_ROWS_HR, col_lo:col_hi], cmap='gray',
          interpolation='nearest', extent=[col_lo, col_hi, ROI1_ROWS_HR.stop, ROI1_ROWS_HR.start])
ax.axvline(PROFILE_COL_HR, color='r', lw=0.8, ls='--', alpha=0.7)
ax.set_title('LR 2x — profile location')
ax.set_xlabel('Column (HR px)')
ax.set_ylabel('Row (HR px)')

plt.suptitle('ROI 1 — Horizontal Bar Resolution (vertical cross-section)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Contrast (modulation) along the vertical cross-section

def local_contrast(profile, window=20):
    """Compute (max - min) / (max + min) in a sliding window."""
    n = len(profile)
    contrast = np.zeros(n)
    hw = window // 2
    for i in range(hw, n - hw):
        seg = profile[i - hw:i + hw]
        mn, mx = seg.min(), seg.max()
        contrast[i] = (mx - mn) / (mx + mn + 1e-9)
    return contrast

fig, ax = plt.subplots(figsize=(12, 4))
for title, img, c in zip(titles, imgs, colors):
    profile = img[ROI1_ROWS_HR, PROFILE_COL_HR]
    ct = local_contrast(profile, window=16)
    ax.plot(rows_hr, ct, lw=1, alpha=0.8, label=title, color=c)

ax.set_ylabel('Local Contrast (Michelson)')
ax.set_xlabel('Row (HR px)')
ax.set_title(f'Local contrast along vertical cross-section at col {PROFILE_COL_HR}')
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## ROI 2 — Slanted Edge MTF

Isolate one edge of a diagonal line for MTF measurement. The ESF (Edge Spread Function) is binned at 4x oversampling, differentiated to get the LSF, and FFT'd to get the MTF.

Note: The red-channel HR pixel pitch is 3.45 µm (sensor) / 2 (upsample) × 2 (Bayer) = 3.45 µm effective.

In [ ]:
# ROI 2 coordinates (HR pixels) — thick diagonal line
# Adjust these based on the full image above
ROI2_HR = (slice(475, 525), slice(640, 690))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, title, img in zip(axes, titles, imgs):
    roi = img[ROI2_HR]
    ax.imshow(roi, cmap='gray', interpolation='nearest')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.suptitle('ROI 2 — Slanted Edge Region', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
def slanted_edge_esf(roi, side='left'):
    """Extract the oversampled ESF from ONE edge of a thick line."""
    from scipy.ndimage import gaussian_filter, sobel

    smooth = gaussian_filter(roi.astype(np.float64), sigma=1.5)
    gy = sobel(smooth, axis=0)
    gx = sobel(smooth, axis=1)
    mag = np.sqrt(gx**2 + gy**2)

    thresh = np.percentile(mag, 85)
    rs, cs = np.where(mag > thresh)
    if len(rs) < 20:
        raise RuntimeError('Too few edge pixels detected')

    row_span = rs.max() - rs.min()
    col_span = cs.max() - cs.min()
    use_row_as_x = (row_span >= col_span)

    if use_row_as_x:
        coeffs_centre = np.polyfit(rs, cs, 1)
        m_c = coeffs_centre[0]
        norm_c = np.sqrt(1 + m_c**2)
        edge_dist = (cs - m_c * rs - coeffs_centre[1]) / norm_c
    else:
        coeffs_centre = np.polyfit(cs, rs, 1)
        m_c = coeffs_centre[0]
        norm_c = np.sqrt(1 + m_c**2)
        edge_dist = (rs - m_c * cs - coeffs_centre[1]) / norm_c

    cluster_mask = edge_dist < 0 if side == 'left' else edge_dist > 0
    rs_sel, cs_sel = rs[cluster_mask], cs[cluster_mask]
    if len(rs_sel) < 10:
        raise RuntimeError(f'Too few edge pixels on {side} side')

    if use_row_as_x:
        coeffs = np.polyfit(rs_sel, cs_sel, 1)
        m = coeffs[0]
        norm = np.sqrt(1 + m**2)
        edge_angle = np.degrees(np.arctan2(1, m))
        nrows, ncols = roi.shape
        rr, cc = np.mgrid[:nrows, :ncols]
        dist = (cc - m * rr - coeffs[1]) / norm
    else:
        coeffs = np.polyfit(cs_sel, rs_sel, 1)
        m = coeffs[0]
        norm = np.sqrt(1 + m**2)
        edge_angle = np.degrees(np.arctan2(m, 1))
        nrows, ncols = roi.shape
        rr, cc = np.mgrid[:nrows, :ncols]
        dist = (rr - m * cc - coeffs[1]) / norm

    flat_dist = dist.ravel()
    flat_val  = roi.ravel().astype(np.float64)
    keep = (flat_dist > -8) & (flat_dist < 10)
    flat_dist = flat_dist[keep]
    flat_val  = flat_val[keep]

    oversample = 4
    bin_width  = 1.0 / oversample
    bins = np.arange(flat_dist.min(), flat_dist.max() + bin_width, bin_width)
    esf_x = 0.5 * (bins[:-1] + bins[1:])
    esf_y = np.empty(len(esf_x))
    esf_y[:] = np.nan
    for i in range(len(esf_x)):
        mask = (flat_dist >= bins[i]) & (flat_dist < bins[i + 1])
        if mask.sum() > 0:
            esf_y[i] = flat_val[mask].mean()

    valid = ~np.isnan(esf_y)
    if valid.sum() > 2:
        esf_y = np.interp(esf_x, esf_x[valid], esf_y[valid])

    if esf_y[-1] < esf_y[0]:
        esf_x = -esf_x[::-1]
        esf_y = esf_y[::-1]

    return esf_x, esf_y, edge_angle


def esf_to_mtf(esf_x, esf_y):
    """ESF -> LSF -> MTF."""
    lsf = np.gradient(esf_y, esf_x)
    window = np.hanning(len(lsf))
    lsf_w  = lsf * window
    n   = len(lsf_w)
    mtf = np.abs(np.fft.fft(lsf_w))
    mtf = mtf[:n // 2]
    if mtf[0] > 0:
        mtf /= mtf[0]
    dx   = np.mean(np.diff(esf_x))
    freq = np.fft.fftfreq(n, d=dx)[:n // 2]
    return freq, mtf, lsf


def mtf_at_fraction(freq, mtf, fraction):
    """Find spatial frequency where MTF crosses a given fraction."""
    above = mtf >= fraction
    if not above.any() or above.all():
        return np.nan
    idx = np.where(np.diff(above.astype(int)) == -1)[0]
    if len(idx) == 0:
        return np.nan
    i = idx[0]
    f0, f1 = freq[i], freq[i + 1]
    m0, m1 = mtf[i], mtf[i + 1]
    if abs(m1 - m0) < 1e-12:
        return f0
    return f0 + (fraction - m0) * (f1 - f0) / (m1 - m0)

print('Slanted-edge MTF functions defined.')

In [ ]:
# Compute ESF and MTF for each method
results = {}
for title, img in zip(titles, imgs):
    print(f'\n{title}:')
    roi = img[ROI2_HR]
    esf_x, esf_y, angle = slanted_edge_esf(roi, side='left')
    freq, mtf, lsf = esf_to_mtf(esf_x, esf_y)
    results[title] = {
        'esf_x': esf_x, 'esf_y': esf_y,
        'freq': freq, 'mtf': mtf, 'lsf': lsf,
        'angle': angle,
    }
print('\nDone.')

In [ ]:
# --- Plot ESF and MTF comparison ---
# Red-channel HR pixel pitch: sensor 3.45um, Bayer 2x, then SR 2x upsample
# Net: each HR pixel = 3.45 um (sensor pitch / SR_upsample * Bayer_factor = 3.45/2*2 = 3.45)
SENSOR_PITCH_MM = 3.45e-3
HR_PITCH_MM     = SENSOR_PITCH_MM  # 3.45 um per red-channel HR pixel

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ESF
ax = axes[0]
for title, c in zip(titles, colors):
    r = results[title]
    ax.plot(r['esf_x'], r['esf_y'], lw=1.2, alpha=0.8, color=c, label=title)
ax.set_xlabel('Distance from edge (px)')
ax.set_ylabel('Intensity')
ax.set_title('Edge Spread Function (ESF)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# MTF
ax = axes[1]
nyquist_cpmm = 1.0 / (2.0 * HR_PITCH_MM)
for title, c in zip(titles, colors):
    r = results[title]
    freq_cpmm = r['freq'] / HR_PITCH_MM
    mtf = r['mtf']
    mask = (freq_cpmm > 0) & (freq_cpmm <= nyquist_cpmm)
    ax.plot(freq_cpmm[mask], mtf[mask], lw=1.5, alpha=0.8, color=c, label=title)

ax.axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.5, label='MTF50')
ax.axhline(0.1, color='gray', ls=':', lw=0.8, alpha=0.5, label='MTF10')
ax.set_xlabel('Spatial frequency (cycles/mm)')
ax.set_ylabel('MTF')
ax.set_title('Modulation Transfer Function')
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)
ax.set_xlim(0, nyquist_cpmm)
ax.grid(True, alpha=0.3)

plt.suptitle('Slanted Edge MTF — RGB (red channel)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- MTF summary table ---
print(f'{"Method":<16s}  {"MTF50 (cy/mm)":>14s}  {"MTF10 (cy/mm)":>14s}')
print('-' * 50)
for title in titles:
    r = results[title]
    freq_cpmm = r['freq'] / HR_PITCH_MM
    m = r['mtf']
    valid = freq_cpmm > 0
    m50 = mtf_at_fraction(freq_cpmm[valid], m[valid], 0.5)
    m10 = mtf_at_fraction(freq_cpmm[valid], m[valid], 0.1)
    print(f'{title:<16s}  {m50:14.1f}  {m10:14.1f}')

## Figure — Original vs SAA+IBP bar comparison

In [ ]:
# Figure: Original vs SAA+IBP — stacked image strips
# Adjust coordinates based on the image above
STRIP_ROWS_HR = slice(650, 800)
STRIP_COLS_HR = slice(1100, 1800)

XSECT_ROWS_HR = slice(700, 730)
XSECT_COL_HR  = 1325

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

ax = axes[0]
ax.imshow(native_2x[STRIP_ROWS_HR, STRIP_COLS_HR], cmap='gray', interpolation='nearest')
ax.text(10, 25, 'Original', fontsize=14, fontweight='bold', color='magenta')
ax.axis('off')

ax = axes[1]
ax.imshow(saa_ibp[STRIP_ROWS_HR, STRIP_COLS_HR], cmap='gray', interpolation='nearest')
ax.text(10, 25, 'SAA + IBP', fontsize=14, fontweight='bold', color='magenta')

import matplotlib.patches as patches
box_r0 = XSECT_ROWS_HR.start - STRIP_ROWS_HR.start
box_r1 = XSECT_ROWS_HR.stop  - STRIP_ROWS_HR.start
box_c0 = XSECT_COL_HR - STRIP_COLS_HR.start - 15
box_c1 = XSECT_COL_HR - STRIP_COLS_HR.start + 15
rect = patches.Rectangle((box_c0, box_r0), box_c1 - box_c0, box_r1 - box_r0,
                           linewidth=2, edgecolor='magenta', facecolor='none')
ax.add_patch(rect)
ax.axis('off')

plt.tight_layout(pad=0.5)
plt.show()